# Giáo trình Dữ liệu lớn – Chương 1

Notebook tổng hợp các đoạn mã trong chương, chạy trên **Google Colab** (ô đầu cài OpenJDK 17, PySpark 3.5.7 và tải kho mã). Trên **Databricks Free Edition**: bỏ ô cài đặt, tải thư mục `data/` lên volume `/Volumes/workspace/default/du_lieu/` do người học tự tạo và thay `data/` bằng đường dẫn này; tính toán serverless của Free Edition không hỗ trợ API RDD/`SparkContext` và `cache()`/`persist()` (xem [README](https://github.com/mocminh/bigdata_code#databricks-free-edition)).

Khác biệt so với sách: đường dẫn `hdfs://.../data/` được đổi thành `data/`, thư mục ghi kết quả là `output/` và `models/`; lệnh `spark.stop()` được đổi thành ghi chú để các ô sau vẫn chạy được. Mã nguyên văn: `code/ch01/doan_ma_*.py`.


In [ ]:
# --- Chuan bi moi truong (Google Colab) ---
# Buoc 1: cai OpenJDK 17 (Spark 3.5 ho tro Java 8/11/17)
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
# Buoc 2: cai dat PySpark tu PyPI (ghim phien ban theo Bang 2.3)
!pip install -q pyspark==3.5.7
if not os.path.exists("data"):
    !git clone -q https://github.com/mocminh/bigdata_code
    %cd bigdata_code
import shutil
for thu_muc in ("output", "models"):          # don ket qua cua lan chay truoc
    shutil.rmtree(thu_muc, ignore_errors=True)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (SparkSession.builder.master("local[*]")
         .appName("GiaoTrinhDuLieuLon-ch01").getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark", spark.version)

## Đoạn mã 1.1. Chương trình đếm từ bằng PySpark.


In [ ]:
from pyspark.sql import SparkSession

# Khoi tao SparkSession - diem vao thong nhat cua ung dung Spark
spark = (SparkSession.builder
         .appName("DemTuPySpark")
         .master("local[*]")
         .getOrCreate())
sc = spark.sparkContext

# Chuoi transformation: chua thuc thi, chi ghi nhan vao DAG
lines = sc.textFile("data/vanban.txt")
words = lines.flatMap(lambda line: line.split(" "))
pairs = words.map(lambda word: (word, 1))
counts = pairs.reduceByKey(lambda a, b: a + b)

# Action collect() kich hoat viec thuc thi toan bo DAG
for word, freq in counts.collect():
    print(word, freq)

# spark.stop()  # giu phien Spark de chay tiep cac o sau

## Đoạn mã 1.2. Chuỗi phép biến đổi thống kê lỗi theo địa chỉ IP.


In [ ]:
logs   = sc.textFile("data/access.log")
errs   = logs.filter(lambda line: "ERROR" in line)
pairs  = errs.map(lambda line: (line.split(" ")[0], 1))
counts = pairs.reduceByKey(lambda a, b: a + b)
result = counts.collect()